# Experiment Setup - Novelty Search Comparisons

This notebook sets up and runs 4 different experimental configurations of the novelty search algorithm:

1. **Pure Random**: Everything is random - random MC, random entities, random verbs (pure chaos!)
2. **Random MC + Associations**: Random main character, but all entities and verbs are associated to it (current default)
3. **Fixed MC + Associations**: Set main character (e.g., "hero" for Zelda), same MC throughout evolution
4. **Max Association MC**: Main character is the entity with maximum number of associations in the logs

Each experiment will be run on different story logs and results will be saved for comparison.

## Experiment Configurations

Now we'll define the four different experimental setups and run them on different story logs.

In [9]:
# Import the main module and setup models
import sys
import os
import json
sys.path.append('..')

from novelty_search_map_elites import (
    setup_models_and_data, 
    novelty_search, 
    map_elites,
    export_archive,
    export_me_archive,
    CN_GRAPH,  # We'll need this for find_max_association_mc
    ALL_SUBJS, ALL_OBJS  # These too
)
from datetime import datetime

# Setup models and data (only need to do this once)
print("Setting up models and data...")
setup_models_and_data(data_path='../bank_files')
print("Setup complete!")

# Function to find the main character with maximum associations
def find_max_association_mc():
    """Find the entity with the maximum number of associations in the ConceptNet graph"""
    if not CN_GRAPH:
        return 'hero'  # fallback
    
    max_assoc_count = 0
    max_mc = 'hero'
    
    for entity, associations in CN_GRAPH.items():
        if entity in ALL_SUBJS:  # Only consider entities that can be subjects
            total_assoc = sum(len(verb_assoc) for verb_assoc in associations.values())
            if total_assoc > max_assoc_count:
                max_assoc_count = total_assoc
                max_mc = entity
    
    print(f"Max association MC found: {max_mc} with {max_assoc_count} associations")
    return max_mc

Setting up models and data...
Loaded data: 5344 subjects, 3191 objects, 1378 verbs
Setup complete!
Loaded data: 5344 subjects, 3191 objects, 1378 verbs
Setup complete!


In [10]:
# Function to read story-specific main characters from file
def read_story_specific_mcs():
    """Read story-specific main characters from logs/ent_mc_names.txt"""
    story_mcs = {}
    try:
        with open('../logs/ent_mc_names.txt', 'r') as f:
            for line in f:
                line = line.strip()
                if line and ' - ' in line:
                    story_name, mcs = line.split(' - ')
                    # Store ALL MC options (they're separated by |)
                    mc_options = [mc.strip() for mc in mcs.split('|')]
                    story_mcs[f'sifted_{story_name}.txt'] = mc_options
                    print(f"Story: {story_name} -> MC options: {mc_options}")
    except FileNotFoundError:
        print("Warning: logs/ent_mc_names.txt not found, using default MCs")
        return {
            'sifted_castle2.txt': ['person'],
            'sifted_drunk_sokoban.txt': ['person'], 
            'sifted_zelda.txt': ['person'],
            'sifted_lock_n_key.txt': ['person']
        }
    return story_mcs

# Experiment 1: Pure Random (Pure Chaos!)
EXP1_PARAMS = {
    'main_char': 'random',
    'other_ents': 'random',
    'mut_main_char': 'random',
    'mut_other_ents': 'random',
    'mut_verbs': 'random',
    'fit_threshold': 0.5,
    'novel_threshold': 0.5,
    'rand_perc': 0.2,
    'num_generations': 20,
    'pop_size': 10
}

# Experiment 2: Random MC + Associations (Default in skeleton)
EXP2_PARAMS = {
    'main_char': 'random',
    'other_ents': 'assoc',
    'mut_main_char': 'random',
    'mut_other_ents': 'assoc',
    'mut_verbs': 'assoc',
    'fit_threshold': 0.5,
    'novel_threshold': 0.5,
    'rand_perc': 0.2,
    'num_generations': 20,
    'pop_size': 10
}

# Experiment 3: Fixed MC + Associations (will set specific MC per story)
EXP3_BASE_PARAMS = {
    'main_char': 'hero',  # Will be overridden per story and MC option
    'other_ents': 'assoc',
    'mut_main_char': 'same',  # Keep same MC throughout evolution
    'mut_other_ents': 'assoc',
    'mut_verbs': 'assoc',
    'fit_threshold': 0.5,
    'novel_threshold': 0.5,
    'rand_perc': 0.2,
    'num_generations': 20,
    'pop_size': 10
}

# Experiment 4: Max Association MC (will find max association MC)
EXP4_PARAMS = {
    'main_char': 'max_assoc',  # Will be determined by find_max_association_mc()
    'other_ents': 'assoc',
    'mut_main_char': 'same',  # Keep same MC throughout evolution
    'mut_other_ents': 'assoc',
    'mut_verbs': 'assoc',
    'fit_threshold': 0.5,
    'novel_threshold': 0.5,
    'rand_perc': 0.2,
    'num_generations': 20,
    'pop_size': 10
}

# Available story files
STORY_FILES = [
    '../sifted_logs/sifted_castle2.txt',
    '../sifted_logs/sifted_zelda.txt',
    '../sifted_logs/sifted_drunk_sokoban.txt',
    '../sifted_logs/sifted_lock_n_key.txt'
]

# Read story-specific MCs from file - now returns all options
STORY_SPECIFIC_MCS = read_story_specific_mcs()

print("Experiment configurations defined!")
print(f"Available stories: {[file.split('/')[-1] for file in STORY_FILES]}")
print(f"Story-specific MCs for Exp 3: {STORY_SPECIFIC_MCS}")

Story: drunk_sokoban -> MC options: ['person', 'human', 'computer', 'soldier']
Story: lock_n_key -> MC options: ['person', 'car', 'dog', 'computer']
Story: castle2 -> MC options: ['person', 'soldier', 'computer']
Story: zelda -> MC options: ['person', 'child', 'dog', 'cat', 'anyone', 'friend']
Experiment configurations defined!
Available stories: ['sifted_castle2.txt', 'sifted_zelda.txt', 'sifted_drunk_sokoban.txt', 'sifted_lock_n_key.txt']
Story-specific MCs for Exp 3: {'sifted_drunk_sokoban.txt': ['person', 'human', 'computer', 'soldier'], 'sifted_lock_n_key.txt': ['person', 'car', 'dog', 'computer'], 'sifted_castle2.txt': ['person', 'soldier', 'computer'], 'sifted_zelda.txt': ['person', 'child', 'dog', 'cat', 'anyone', 'friend']}


In [11]:
# Function to run a single experiment
def run_experiment(exp_name, story_file, params, specific_mc=None):
    ''' Run a single experiment and save results 
    
    Args:
        exp_name: Name of experiment
        story_file: Path to story file
        params: Experiment parameters
        specific_mc: Optional specific main character to use (for Exp3 variations)
    '''
    print(f"\n{'='*60}")
    print(f"RUNNING EXPERIMENT: {exp_name}")
    if specific_mc:
        print(f"Specific MC: {specific_mc}")
    print(f"Story: {story_file.split('/')[-1]}")
    print(f"Parameters: {params}")
    print(f"{'='*60}")
    
    # Handle special MC configurations
    final_params = params.copy()
    if params['main_char'] == 'max_assoc':
        max_mc = find_max_association_mc()
        final_params['main_char'] = max_mc
    elif specific_mc:
        # Use the specific MC provided (for Exp3 variations)
        final_params['main_char'] = specific_mc
    elif exp_name == "Exp3_Fixed_MC":
        story_name = story_file.split('/')[-1]
        if story_name in STORY_SPECIFIC_MCS:
            # Use first MC option as default if no specific_mc provided
            final_params['main_char'] = STORY_SPECIFIC_MCS[story_name][0]
    
    print(f"Final MC for this run: {final_params['main_char']}")
    print("-" * 60)
    
    # Run the experiment
    try:
        archive, best_fic, story = novelty_search(story_file, params=final_params)
        
        # Create output filenames
        timestamp = datetime.now().strftime("[%m-%d-%Y_%H%M]")
        story_name = story_file.split('/')[-1].replace('.txt', '')
        
        # Add MC suffix to experiment name if specific MC is used
        exp_name_with_mc = exp_name
        if specific_mc:
            exp_name_with_mc = f"{exp_name}_{specific_mc}"
        
        # Create directories if they don't exist
        exp_dir = f"../experiment_results/{exp_name}"
        os.makedirs(f"{exp_dir}/archives", exist_ok=True)
        os.makedirs(f"{exp_dir}/genomes", exist_ok=True)
        os.makedirs(f"{exp_dir}/stories", exist_ok=True)
        
        # Save results
        genome_file = f"{exp_dir}/genomes/{story_name}_{exp_name_with_mc}_{timestamp}.json"
        archive_file = f"{exp_dir}/archives/{story_name}_{exp_name_with_mc}_{timestamp}.json"
        story_file_out = f"{exp_dir}/stories/{story_name}_{exp_name_with_mc}_{timestamp}.txt"
        
        # Export results
        best_fic.export_fic(story, out_file=genome_file)
        export_archive(archive, out_file=archive_file, story=story)
        best_fic.generate_story(story, out_file=story_file_out)
        
        print(f"\nEXPERIMENT COMPLETED SUCCESSFULLY!")
        print(f"Final archive size: {len(archive)}")
        print(f"Best fitness achieved: {best_fic.fitness:.4f}")
        print(f"Best MC: {best_fic.mc}")
        print(f"Results saved to: {exp_dir}")
        
        return {
            'exp_name': exp_name_with_mc,
            'story_file': story_file,
            'main_char': final_params['main_char'],
            'archive_size': len(archive),
            'best_fitness': best_fic.fitness,
            'best_mc': best_fic.mc,
            'genome_file': genome_file,
            'archive_file': archive_file,
            'story_file_out': story_file_out
        }
        
    except Exception as e:
        print(f"ERROR in experiment {exp_name}: {e}")
        return None

print("Single experiment runner defined!")

Single experiment runner defined!


In [20]:
# Function to run a single experiment
def run_experiment(exp_name, story_file, params, specific_mc=None, algorithm='both'):
    ''' Run a single experiment and save results 
    
    Args:
        exp_name: Name of experiment
        story_file: Path to story file
        params: Experiment parameters
        specific_mc: Optional specific main character to use (for Exp3 variations)
        algorithm: Which algorithm to run ('novelty_search', 'map_elites', or 'both')
    '''
    print(f"\n{'='*60}")
    print(f"RUNNING EXPERIMENT: {exp_name}")
    if specific_mc:
        print(f"Specific MC: {specific_mc}")
    print(f"Story: {story_file.split('/')[-1]}")
    print(f"Algorithm: {algorithm}")
    print(f"Parameters: {params}")
    print(f"{'='*60}")
    
    # Handle special MC configurations
    final_params = params.copy()
    if params['main_char'] == 'max_assoc':
        max_mc = find_max_association_mc()
        final_params['main_char'] = max_mc
    elif specific_mc:
        # Use the specific MC provided (for Exp3 variations)
        final_params['main_char'] = specific_mc
    elif exp_name == "Exp3_Fixed_MC":
        story_name = story_file.split('/')[-1]
        if story_name in STORY_SPECIFIC_MCS:
            # Use first MC option as default if no specific_mc provided
            final_params['main_char'] = STORY_SPECIFIC_MCS[story_name][0]
    
    print(f"Final MC for this run: {final_params['main_char']}")
    print("-" * 60)
    
    results = []
    algorithms_to_run = []
    
    if algorithm == 'both':
        algorithms_to_run = [('novelty_search', novelty_search), ('map_elites', map_elites)]
    elif algorithm == 'novelty_search':
        algorithms_to_run = [('novelty_search', novelty_search)]
    elif algorithm == 'map_elites':
        algorithms_to_run = [('map_elites', map_elites)]
    else:
        raise ValueError(f"Unknown algorithm: {algorithm}")
    
    # Ensure default output directories exist
    os.makedirs('../novelty_search_out/archive', exist_ok=True)
    os.makedirs('../map_elites_out/archive', exist_ok=True)
    
    # Run each algorithm
    for alg_name, alg_func in algorithms_to_run:
        print(f"\nRunning {alg_name.upper()}...")
        
        try:
            # Run the algorithm
            archive, best_fic, story = alg_func(story_file, params=final_params)
            
            # Create output filenames
            timestamp = datetime.now().strftime("[%m-%d-%Y_%H%M]")
            story_name = story_file.split('/')[-1].replace('.txt', '')
            
            # Add MC suffix to experiment name if specific MC is used
            exp_name_with_mc = exp_name
            if specific_mc:
                exp_name_with_mc = f"{exp_name}_{specific_mc}"
            
            # Create directory structure for our custom organization
            exp_dir = f"../experiment_results/{exp_name}_{alg_name}"
            os.makedirs(f"{exp_dir}/archives", exist_ok=True)
            os.makedirs(f"{exp_dir}/genomes", exist_ok=True)
            os.makedirs(f"{exp_dir}/stories", exist_ok=True)
            
            # Create filenames (just names, not full paths for export functions)
            base_filename = f"{story_name}_{exp_name_with_mc}_{timestamp}"
            genome_filename = f"{base_filename}.json"
            archive_filename = f"{base_filename}.json"
            story_filename = f"{base_filename}.txt"
            
            # Full paths for our organized output
            genome_file = f"{exp_dir}/genomes/{genome_filename}"
            archive_file = f"{exp_dir}/archives/{archive_filename}"
            story_file_out = f"{exp_dir}/stories/{story_filename}"
            
            # Export best genome (this works with full paths)
            best_fic.export_fic(story, out_file=genome_file)
            
            # Export archive using the export functions with correct output directories
            import shutil
            if alg_name == 'novelty_search':
                # Use the export function with correct output directory
                export_archive(archive, out_file=archive_filename, story=story, output_dir='../novelty_search_out/archive/')
                # Copy to our organized structure
                default_archive_path = f"../novelty_search_out/archive/{archive_filename}"
                if os.path.exists(default_archive_path):
                    shutil.copy2(default_archive_path, archive_file)
            else:  # map_elites
                # Use the export function with correct output directory
                export_me_archive(archive, out_file=archive_filename, story=story, output_dir='../map_elites_out/archive/')
                # Copy to our organized structure
                default_archive_path = f"../map_elites_out/archive/{archive_filename}"
                if os.path.exists(default_archive_path):
                    shutil.copy2(default_archive_path, archive_file)
            
            # Generate story output
            best_fic.generate_story(story, out_file=story_file_out)
            
            print(f"\n{alg_name.upper()} COMPLETED SUCCESSFULLY!")
            if alg_name == 'novelty_search':
                print(f"Final archive size: {len(archive)}")
            else:  # map_elites
                print(f"Final archive size: {len(archive)} cells")
            print(f"Best fitness achieved: {best_fic.fitness:.4f}")
            print(f"Best MC: {best_fic.mc}")
            print(f"Results saved to: {exp_dir}")
            
            result = {
                'exp_name': exp_name_with_mc,
                'algorithm': alg_name,
                'story_file': story_file,
                'main_char': final_params['main_char'],
                'archive_size': len(archive),
                'best_fitness': best_fic.fitness,
                'best_mc': best_fic.mc,
                'genome_file': genome_file,
                'archive_file': archive_file,
                'story_file_out': story_file_out
            }
            results.append(result)
            
        except Exception as e:
            print(f"ERROR in {alg_name} for experiment {exp_name}: {e}")
            import traceback
            traceback.print_exc()
    
    return results if results else None

print("Single experiment runner defined!")

Single experiment runner defined!


In [21]:
# Test with a small experiment first
TEST_PARAMS = {
    'main_char': 'random',
    'other_ents': 'assoc',
    'mut_main_char': 'random',
    'mut_other_ents': 'assoc',
    'mut_verbs': 'assoc',
    'fit_threshold': 0.5,
    'novel_threshold': 0.5,
    'rand_perc': 0.2,
    'num_generations': 2,  # Very small for testing
    'pop_size': 5          # Very small for testing
}

# Test the find_max_association_mc function
max_mc = find_max_association_mc()
print(f"Found max association MC: {max_mc}")

# Quick test run of one experiment with one algorithm
print("\n" + "="*60)
print("RUNNING TEST EXPERIMENT")
print("="*60)

# Test novelty search only first
test_results = run_experiment(
    exp_name="Test_Run", 
    story_file=STORY_FILES[0], 
    params=TEST_PARAMS,
    algorithm='novelty_search'  # Test just novelty search first
)

if test_results:
    print("✓ Test run completed successfully!")
    for result in test_results:
        print(f"  - Algorithm: {result['algorithm']}")
        print(f"  - Best fitness: {result['best_fitness']:.4f}")
        print(f"  - Archive size: {result['archive_size']}")
else:
    print("✗ Test run failed!")

Max association MC found: person with 599 associations
Found max association MC: person

RUNNING TEST EXPERIMENT

RUNNING EXPERIMENT: Test_Run
Story: sifted_castle2.txt
Algorithm: novelty_search
Parameters: {'main_char': 'random', 'other_ents': 'assoc', 'mut_main_char': 'random', 'mut_other_ents': 'assoc', 'mut_verbs': 'assoc', 'fit_threshold': 0.5, 'novel_threshold': 0.5, 'rand_perc': 0.2, 'num_generations': 2, 'pop_size': 5}
Final MC for this run: random
------------------------------------------------------------

Running NOVELTY_SEARCH...
Generation 1 / 2 -- [Overall Best Fitness: 0.000 | Archive Size: 0]
  Pop Fitness: max 0.463, min 0.343, avg 0.397
   FicGenome of best population individual:
     - Best MC: coloring
     - Best Ents: ['coloring2', 'coloring', 'coloring2', 'water', 'water', 'crayon', 'crayon', 'crayon2', 'water2', 'crayon3', 'water2', 'crayon4', 'water3', 'crayon5', 'crayon6', 'crayon7', 'crayon8', 'crayon9']
     - Best Verbs: {'added', 'bundled', 'did', 'gave',

In [22]:
# Main function to run all experiments
def run_all_experiments(algorithm='both'):
    ''' Run all four experiments on all story files for both algorithms
    For Exp3_Fixed_MC, runs separate experiments for each main character option
    
    Args:
        algorithm: Which algorithm to run ('novelty_search', 'map_elites', or 'both')
    '''
    
    experiments = [
        ("Exp1_Pure_Random", EXP1_PARAMS),
        ("Exp2_Random_MC_Assoc", EXP2_PARAMS),
        ("Exp3_Fixed_MC", EXP3_BASE_PARAMS),
        ("Exp4_Max_Assoc_MC", EXP4_PARAMS)
    ]
    
    all_results = []
    
    # Calculate total experiments
    total_exp3_runs = sum(len(STORY_SPECIFIC_MCS[story.split('/')[-1]]) 
                          for story in STORY_FILES 
                          if story.split('/')[-1] in STORY_SPECIFIC_MCS)
    total_other_runs = (len(experiments) - 1) * len(STORY_FILES)  # -1 because Exp3 is special
    total_base_runs = total_other_runs + total_exp3_runs
    
    # Multiply by number of algorithms
    algorithms_count = 2 if algorithm == 'both' else 1
    total_runs = total_base_runs * algorithms_count
    
    print(f"Starting comprehensive experiment suite...")
    print(f"Running {len(experiments)} experiment types on {len(STORY_FILES)} stories")
    print(f"Algorithms to run: {algorithm}")
    print(f"Exp3_Fixed_MC will run {total_exp3_runs} variations (multiple MCs per story)")
    print(f"Total experiments: {total_runs}")
    
    for exp_name, exp_params in experiments:
        if exp_name == "Exp3_Fixed_MC":
            # Special handling for Exp3 - run multiple experiments per story
            print(f"\n{'='*80}")
            print(f"STARTING EXP3_FIXED_MC WITH MULTIPLE MAIN CHARACTERS")
            print(f"{'='*80}")
            
            for story_file in STORY_FILES:
                story_name = story_file.split('/')[-1]
                if story_name in STORY_SPECIFIC_MCS:
                    mc_options = STORY_SPECIFIC_MCS[story_name]
                    print(f"\nRunning Exp3 for {story_name} with {len(mc_options)} main characters: {mc_options}")
                    
                    for mc in mc_options:
                        results = run_experiment(exp_name, story_file, exp_params, 
                                               specific_mc=mc, algorithm=algorithm)
                        if results:
                            all_results.extend(results)
                else:
                    print(f"Warning: No MCs found for {story_name}, skipping Exp3")
        else:
            # Standard handling for other experiments
            for story_file in STORY_FILES:
                results = run_experiment(exp_name, story_file, exp_params, algorithm=algorithm)
                if results:
                    all_results.extend(results)
    
    # Save summary of all results
    timestamp = datetime.now().strftime("[%m-%d-%Y_%H%M]")
    summary_file = f"../experiment_results/experiment_summary_{algorithm}_{timestamp}.json"
    os.makedirs("../experiment_results", exist_ok=True)
    
    with open(summary_file, 'w') as f:
        json.dump(all_results, f, indent=3)
    
    print(f"\n{'='*80}")
    print(f"ALL EXPERIMENTS COMPLETED!")
    print(f"{'='*80}")
    print(f"Total successful runs: {len(all_results)}")
    print(f"Summary saved to: {summary_file}")
    
    # Print summary table
    print(f"\nEXPERIMENT SUMMARY:")
    print(f"{'Experiment':<25} {'Algorithm':<15} {'Story':<15} {'Main Char':<10} {'Archive Size':<12} {'Best Fitness':<12}")
    print("-" * 110)
    for result in all_results:
        story_name = result['story_file'].split('/')[-1].replace('.txt', '')
        print(f"{result['exp_name']:<25} {result['algorithm']:<15} {story_name:<15} {result['main_char']:<10} {result['archive_size']:<12} {result['best_fitness']:<12.4f}")
    
    return all_results

print("Main experiment runner defined!")

Main experiment runner defined!


In [23]:
# Test both algorithms on one experiment
print("\n" + "="*60)
print("TESTING BOTH ALGORITHMS")
print("="*60)

test_results_both = run_experiment(
    exp_name="Test_Both", 
    story_file=STORY_FILES[0], 
    params=TEST_PARAMS,
    algorithm='both'  # Test both algorithms
)

if test_results_both and len(test_results_both) == 2:
    print("✓ Both algorithms completed successfully!")
    for result in test_results_both:
        print(f"  - {result['algorithm']}: Best fitness {result['best_fitness']:.4f}, Archive size {result['archive_size']}")
else:
    print("✗ Test failed - expected 2 results, got", len(test_results_both) if test_results_both else 0)


TESTING BOTH ALGORITHMS

RUNNING EXPERIMENT: Test_Both
Story: sifted_castle2.txt
Algorithm: both
Parameters: {'main_char': 'random', 'other_ents': 'assoc', 'mut_main_char': 'random', 'mut_other_ents': 'assoc', 'mut_verbs': 'assoc', 'fit_threshold': 0.5, 'novel_threshold': 0.5, 'rand_perc': 0.2, 'num_generations': 2, 'pop_size': 5}
Final MC for this run: random
------------------------------------------------------------

Running NOVELTY_SEARCH...
Generation 1 / 2 -- [Overall Best Fitness: 0.000 | Archive Size: 0]
  Pop Fitness: max 0.474, min 0.360, avg 0.418
   FicGenome of best population individual:
     - Best MC: cooking
     - Best Ents: ['cooking2', 'cooking', 'cooking2', 'dinner', 'consistency', 'food', 'frying', 'food2', 'dinner2', 'food3', 'consistency2', 'food4', 'consistency3', 'food5', 'food6', 'food7', 'food8', 'food9']
     - Best Verbs: {'rationalized', 'included', 'powered', 'warmed', 'flattened', 'perpetuated', 'ate', 'obscured', 'caked', 'cooked', 'eliminated', 'tos

## Execute Full Experiments

Now you're ready to run the complete experiment suite! Choose one of the options below:

### Option 1: Run Both Algorithms (Comprehensive)
This runs all 4 experiments on all 4 story files using both Novelty Search and MAP-Elites.
- **Estimated time**: 2-4 hours (depending on your system)
- **Total runs**: ~32+ experiments (Exp3 runs multiple times per story)

### Option 2: Run Single Algorithm
Run all experiments with just one algorithm:
- **Novelty Search only**: Faster, focuses on diversity
- **MAP-Elites only**: Faster, focuses on fitness + diversity archive

### Option 3: Test Run (Recommended First)
Run a smaller subset to test the pipeline before committing to the full experiment suite.

In [24]:
# OPTION 1: Run BOTH algorithms (Comprehensive experiment suite)
# Uncomment the line below to run the full suite:
# results = run_all_experiments(algorithm='both')

print("To run the full experiment suite with both algorithms, uncomment the line above.")
print("WARNING: This will take several hours to complete!")

To run the full experiment suite with both algorithms, uncomment the line above.


In [25]:
# OPTION 2A: Run NOVELTY SEARCH only
# Uncomment the line below to run novelty search experiments:
# results_ns = run_all_experiments(algorithm='novelty_search')

print("To run only Novelty Search experiments, uncomment the line above.")

To run only Novelty Search experiments, uncomment the line above.


In [26]:
# OPTION 2B: Run MAP-ELITES only
# Uncomment the line below to run MAP-Elites experiments:
# results_me = run_all_experiments(algorithm='map_elites')

print("To run only MAP-Elites experiments, uncomment the line above.")

To run only MAP-Elites experiments, uncomment the line above.


In [27]:
# OPTION 3: Test run with smaller parameters (Recommended first!)
# Run a subset to test everything works
TEST_EXPERIMENTS = [
    ("Exp1_Pure_Random", EXP1_PARAMS),
    ("Exp2_Random_MC_Assoc", EXP2_PARAMS)
]

def run_test_experiments(algorithm='both', num_stories=2):
    """Run a smaller test of the experiment pipeline"""
    print(f"Running TEST experiments with {algorithm} algorithm(s) on {num_stories} stories...")
    
    all_results = []
    test_stories = STORY_FILES[:num_stories]  # Just use first N stories
    
    for exp_name, exp_params in TEST_EXPERIMENTS:
        for story_file in test_stories:
            results = run_experiment(exp_name, story_file, exp_params, algorithm=algorithm)
            if results:
                all_results.extend(results)
    
    print(f"\nTest completed! {len(all_results)} experiments run.")
    return all_results

# Uncomment the line below to run test experiments:
# test_results = run_test_experiments(algorithm='both', num_stories=2)

print("To run a quick test, uncomment the line above.")

To run a quick test, uncomment the line above.


## Summary

✅ **Setup Complete!** You now have a comprehensive experiment framework that can run:

### Four Experiment Types:
1. **Pure Random**: Everything is random (pure chaos!)
2. **Random MC + Associations**: Random main character, associated entities/verbs
3. **Fixed MC + Associations**: Fixed main character per story, associated entities/verbs  
4. **Max Association MC**: Main character with maximum associations, associated entities/verbs

### Two Algorithms:
- **Novelty Search**: Focuses on diversity and exploration
- **MAP-Elites**: Maintains archive of diverse high-fitness solutions

### Four Story Files:
- `sifted_castle2.txt`
- `sifted_zelda.txt` 
- `sifted_drunk_sokoban.txt`
- `sifted_lock_n_key.txt`

### Results Organization:
All results are saved in `../experiment_results/` with organized subdirectories:
- `{ExpName}_{Algorithm}/archives/` - Algorithm archives
- `{ExpName}_{Algorithm}/genomes/` - Best genome files
- `{ExpName}_{Algorithm}/stories/` - Generated story outputs

**Next Steps:**
1. Run Option 3 (test run) first to verify everything works
2. Choose your preferred algorithm(s) and run the full experiment suite
3. Analyze results using the generated summary JSON files

The framework is flexible - you can easily modify parameters, add new experiments, or run subsets of the full suite!